In [1]:
import sys
import pandas as pd
import logging
sys.path.append('/Users/BKieft/Metabolomics/metatlas2/metatlas2')
import database_interact as dbi
import pubchem_retrieval as pcr
import load_tools as ldt
import lcmsruns_tools as lrt
import ms1_ms2_analysis as msa
import rt_align_tools as rat
import targeted_analysis as tga
import targeted_gui as tgi
import logging_config as lcf
#import spectrum_handlers as sph

logger = lcf.setup_logging(log_level=logging.INFO)
pd.options.display.max_colwidth = 300

In [2]:
# Required: load config file
CONFIG = ldt.load_metatlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")
# Required: table with compounds to add to db. Minimally, must have columns 'inchi_key' and 'label'
COMPOUND_INPUTS = ["/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv", 
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_NEG.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_POS.tsv",
                   "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_EMA_NEG.tsv"]
# Required: table of compounds; minimally, must have columns 'inchi_key', 'chromatography', 'polarity', 'rt_peak', 'mz', 'adduct'
ATLAS_INPUTS = ["/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_ISTD_POS.tsv",
                "/Users/BKieft/Metabolomics/metatlas2/data/atlas_tables/HILICZ/HILICZ_QC_POS.tsv"]
# Required: name of atlas (does not have to be unique) 
ATLAS_NAMES = ["HILIC Positive ISTD Default", "HILIC Positive QC Default"]
# Optional: description of atlas
ATLAS_DESC = [f"Default ISTD compounds for HILIC Positive", f"Default QC compounds for HILIC Positive"]
# Required: Project name
PROJECT_NAME = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"
# Required: paths based on config
PROJECT_FILES_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/lcmsruns"
PROJECT_DB_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/{PROJECT_NAME}.duckdb"
# Required: path to id report
ANALYSIS_OUTPUT_PATH = f'/Users/BKieft/Downloads/{PROJECT_NAME}'

In [3]:
new_main_database = False
new_atlases = False
new_rt_alignment = False
new_analysis = True

In [4]:
if new_main_database is True:
    dbi.create_metatlas_database(CONFIG)
    for COMPOUNDS_INPUT in COMPOUND_INPUTS:
        compounds_df = ldt.load_compound_input(COMPOUNDS_INPUT)
        pcr.retrieve_pubchem_info(compounds_df, CONFIG)
        dbi.add_compounds_to_db(compounds_df, CONFIG, COMPOUNDS_INPUT)
    dbi.validate_database(CONFIG)

In [5]:
if new_atlases is True:
    new_atlas_info = ()
    for ATLAS_INPUT, ATLAS_NAME, ATLAS_DESC in zip(ATLAS_INPUTS, ATLAS_NAMES, ATLAS_DESC):
        atlas_compounds_df = ldt.load_atlas_input(ATLAS_INPUT)
        atlas_uid, atlas_name = dbi.create_atlas_from_compounds(atlas_compounds_df, ATLAS_NAME, ATLAS_DESC, CONFIG)
        new_atlas_info += ((atlas_uid, atlas_name),)
    dbi.validate_database(CONFIG)
    QC_ATLAS_UID = next(uid for uid, name in new_atlas_info if "QC" in name)
    TARGET_ATLAS_UID = next(uid for uid, name in new_atlas_info if "QC" not in name)
else:
    QC_ATLAS_UID = "atl-raw-7906f4bf395d4ed791f3763ed10f61c8"
    TARGET_ATLAS_UID = "atl-raw-2e24a9969a6a4758844b99162e9e0a03"
    logger.info(f"Skipping new atlas creation and using QC Atlas: {QC_ATLAS_UID} and TARGET Atlas: {TARGET_ATLAS_UID}")

2025-09-04 21:59:30 | metatlas2 | INFO | Skipping new atlas creation and using QC Atlas: atl-raw-7906f4bf395d4ed791f3763ed10f61c8 and TARGET Atlas: atl-raw-2e24a9969a6a4758844b99162e9e0a03


In [6]:
if new_rt_alignment is True:
    dbi.create_project_database(PROJECT_DB_PATH, CONFIG)
    lcmsrun_files = dbi.save_lcmsruns_to_db(PROJECT_DB_PATH, PROJECT_NAME, PROJECT_FILES_PATH, CONFIG)
    matches_df, matching_stats = msa.extract_and_match_qc_compounds(PROJECT_DB_PATH, QC_ATLAS_UID, CONFIG)
    best_model, model_results, model_stats = rat.build_rt_alignment_model(matches_df, CONFIG)
    ANALYSIS_ATLAS_UID = rat.apply_rt_correction_to_target(PROJECT_DB_PATH, TARGET_ATLAS_UID, CONFIG, best_model, lcmsrun_files, model_results)
else:
    ANALYSIS_ATLAS_UID = "atl-rta-0c89e582233e481ca4f1920611f6007f"
    logger.info(f"Skipping new RT alignment and using {ANALYSIS_ATLAS_UID}")

2025-09-04 21:59:30 | metatlas2 | INFO | Skipping new RT alignment and using atl-rta-0c89e582233e481ca4f1920611f6007f


In [7]:
if new_analysis is True:
    atlas_dataframe, project_data, plot_data = tga.run_targeted_analysis_workflow(PROJECT_DB_PATH, ANALYSIS_ATLAS_UID, CONFIG)
    gui = tgi.create_gui(plot_data, CONFIG, PROJECT_DB_PATH)
    gui._project_data = project_data
    display(gui)

2025-09-04 21:59:30 | metatlas2.targeted_analysis | INFO | Setting up targeted analysis database...
2025-09-04 21:59:30 | metatlas2.targeted_analysis | INFO | Running fresh targeted analysis...
2025-09-04 21:59:30 | metatlas2.targeted_analysis | INFO | Loading target atlas...
2025-09-04 21:59:30 | metatlas2.database_interact | INFO | Retrieved 65 compounds from project database for atlas: HILIC Positive ISTD Default (RT Corrected) (atl-rta-0c89e582233e481ca4f1920611f6007f)
2025-09-04 21:59:30 | metatlas2.database_interact | INFO | RT-corrected atlas detected with experimental data
2025-09-04 21:59:30 | metatlas2.database_interact | INFO | Enriched 65 compounds with metadata from main database
2025-09-04 21:59:30 | metatlas2.targeted_analysis | INFO | Created Atlas dataframe with 65 compounds
2025-09-04 21:59:30 | metatlas2.targeted_analysis | INFO | Loading experimental files from project database...
2025-09-04 21:59:30 | metatlas2.database_interact | INFO | Found 6 experimental files 

Processing files in parallel:   0%|          | 0/6 [00:00<?, ?it/s]

2025-09-04 21:59:37 | metatlas2.ms1_ms2_analysis | INFO | Extraction complete:
2025-09-04 21:59:37 | metatlas2.ms1_ms2_analysis | INFO |   Total compounds: 60
2025-09-04 21:59:37 | metatlas2.ms1_ms2_analysis | INFO |   Compounds with EIC data: 60
2025-09-04 21:59:37 | metatlas2.ms1_ms2_analysis | INFO |   Compounds with MS2 data: 38
2025-09-04 21:59:37 | metatlas2.ms1_ms2_analysis | INFO |   Compounds with MS2 hits: 37
2025-09-04 21:59:37 | metatlas2.ms1_ms2_analysis | INFO |   Total EIC traces: 284
2025-09-04 21:59:37 | metatlas2.ms1_ms2_analysis | INFO |   Total MS2 spectra: 352
2025-09-04 21:59:37 | metatlas2.simple_cache | INFO | Data cache saved to /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/cache/data_cache/2025-09-04T21-59-37-401996
2025-09-04 21:59:37 | metatlas2.targeted_analysis | INFO | Saved data cache with timestamp: 2025-09-04T21:59:37.401996
2025-09-04 21:59:37 | metatlas2.targeted_gui | IN

In [9]:
inchi_key = 'AGPKZVBTJJNPAG-WHFBIAKZSA-N'
mods = gui.get_modifications()
print(mods.get_rt_bounds(inchi_key))
print(mods.get_annotations(inchi_key))

None
{}


In [10]:
if new_analysis is True:
    post_analysis_atlas_uid, targeted_analysis_uid, comprehensive_report = tga.run_post_analysis_workflow(
        project_db_path=PROJECT_DB_PATH,
        analysis_atlas_uid=ANALYSIS_ATLAS_UID,
        gui_container=gui,
        atlas_dataframe=atlas_dataframe,
        project_name=PROJECT_NAME,
        config=CONFIG,
        analysis_output_path=ANALYSIS_OUTPUT_PATH
    )

2025-09-04 21:59:37 | metatlas2.targeted_analysis | INFO | Starting post-analysis workflow...
2025-09-04 21:59:37 | metatlas2.targeted_analysis | INFO | Creating post-analysis atlas...
2025-09-04 21:59:37 | metatlas2.simple_cache | INFO | GUI cache (complete) saved to /Users/BKieft/Metabolomics/metatlas2/data/test_data/projects/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519/cache/gui_cache/complete/2025-09-04T21-59-37-948994
2025-09-04 21:59:37 | metatlas2.targeted_analysis | INFO | Saved final AnalystModifications to complete cache: 2025-09-04T21:59:37.948994
2025-09-04 21:59:37 | metatlas2.database_interact | INFO | Retrieved 65 compounds from project database for atlas: HILIC Positive ISTD Default (RT Corrected) (atl-rta-0c89e582233e481ca4f1920611f6007f)
2025-09-04 21:59:37 | metatlas2.database_interact | INFO | RT-corrected atlas detected with experimental data
2025-09-04 21:59:37 | metatlas2.database_interact | INFO | Enriched 65 compounds with metadata from main

Processing compounds:   0%|          | 0/60 [00:00<?, ?it/s]

2025-09-04 21:59:38 | metatlas2.database_interact | INFO | Report saved to /Users/BKieft/Downloads/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519
2025-09-04 21:59:38 | metatlas2.targeted_analysis | INFO | Generated comprehensive report and saved to /Users/BKieft/Downloads/20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519
2025-09-04 21:59:38 | metatlas2.targeted_analysis | INFO | Post-analysis workflow completed successfully


In [11]:
dbi.validate_database(CONFIG, PROJECT_DB_PATH)

2025-09-04 21:59:38 | metatlas2.database_interact | INFO | Database Validation:
2025-09-04 21:59:38 | metatlas2.database_interact | INFO |    Atlases: 7
2025-09-04 21:59:38 | metatlas2.database_interact | INFO |    Targeted analyses: 6
2025-09-04 21:59:38 | metatlas2.database_interact | INFO |    RT alignment models: 1
2025-09-04 21:59:38 | metatlas2.database_interact | INFO |    Experimental RT/MZ entries: 65
2025-09-04 21:59:38 | metatlas2.database_interact | INFO |    Available atlases:
2025-09-04 21:59:38 | metatlas2.database_interact | INFO |       atl-rta-0c89e582233e481ca4f1920611f6007f
2025-09-04 21:59:38 | metatlas2.database_interact | INFO |       atl-tga-5957c5649496448c9480b10032eb4247
2025-09-04 21:59:38 | metatlas2.database_interact | INFO |       atl-tga-c04a5ddd305e4e6eb96ae3fbbc95b3ba
2025-09-04 21:59:38 | metatlas2.database_interact | INFO |       atl-tga-d682e64f7145441593a2efd38e5ef480
2025-09-04 21:59:38 | metatlas2.database_interact | INFO |       atl-tga-18863eb3